# Experiment: Depth Edge Boundary Refinement

Compare conservative boundary-aware depth refinement strategies for existing DINO+SAM2 masks.

The original SAM2 mask is always the default. Depth is used only near mask boundaries as a candidate cleanup/review signal.

## 1. Setup and Config

In [ ]:
from __future__ import annotations

import json, math, random, sys
from collections import Counter
from pathlib import Path
from typing import Any

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

try:
    from IPython.display import display
except Exception:
    display = print


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / "notebooks").exists() and (p / "outputs").exists():
            return p
    return start


REPO_ROOT = find_repo_root()


def repo_path(value: str | Path | None) -> Path | None:
    if value is None or str(value).strip() == "":
        return None
    p = Path(value)
    return p if p.is_absolute() else REPO_ROOT / p


CONFIG = {
    "PREVIOUS_EXP_DIR": "outputs/exp_depth_effect_on_sam_masks",
    "OUTPUT_DIR": "outputs/exp_depth_edge_boundary_refinement",
    "USE_PREVIOUS_SAMPLES": True,
    "FRAMES_DIR": None,
    "PROPOSALS_JSON": None,
    "DEPTH_DIR": None,
    "SAMPLE_N": 12,
    "RANDOM_SEED": 42,
    "CORE_ERODE_KERNEL": 7,
    "BAND_DILATE_KERNEL": 11,
    "MIN_MASK_AREA": 100,
    "MIN_COMPONENT_AREA": 80,
    "DEPTH_EDGE_METHOD": "sobel",
    "DEPTH_EDGE_PERCENTILE": 80,
    "DEPTH_EDGE_SMOOTH_KSIZE": 5,
    "ENABLE_BOUNDARY_ONLY_TRIM": True,
    "BOUNDARY_DEPTH_OUTLIER_THRESHOLD": 0.12,
    "BOUNDARY_TRIM_MAX_REMOVED_RATIO": 0.20,
    "BOUNDARY_TRIM_MIN_FINAL_AREA_RATIO": 0.80,
    "ENABLE_DEPTH_EDGE_SNAPPING": True,
    "SNAP_SEARCH_RADIUS": 6,
    "SNAP_MAX_AREA_CHANGE_RATIO": 0.20,
    "SNAP_MIN_EDGE_ALIGNMENT_GAIN": 0.05,
    "ENABLE_GRABCUT_CANDIDATE": True,
    "GRABCUT_ITER": 5,
    "ENABLE_RANDOM_WALKER_CANDIDATE": False,
    "ENABLE_SAM2_RERUN": False,
    "SAM2_CONFIG": None,
    "SAM2_CHECKPOINT": None,
    "FINAL_MASK_POLICY": "conservative",
    "MAX_ALLOWED_AREA_DROP": 0.20,
    "MAX_ALLOWED_COMPONENT_INCREASE": 1,
    "MIN_EDGE_ALIGNMENT_FOR_ACCEPT": 0.45,
    "SAVE_VIS": True,
    "SHOW_INLINE": True,
    "SAVE_CONTACT_SHEET": True,
}

PREV_DIR = repo_path(CONFIG["PREVIOUS_EXP_DIR"])
OUTPUT_DIR = repo_path(CONFIG["OUTPUT_DIR"])
assert PREV_DIR is not None and OUTPUT_DIR is not None
DIRS = {name: OUTPUT_DIR / name for name in [
    "masks_original", "masks_aggressive_baseline", "masks_boundary_trim", "masks_edge_snap",
    "masks_grabcut", "masks_random_walker", "masks_final", "visualizations", "debug"
]}
for p in DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

print("repo root:", REPO_ROOT)
print(json.dumps(CONFIG, indent=2, ensure_ascii=False))

## 2. Load Previous Notebook Outputs

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def read_json_or_jsonl(path: Path):
    if path.suffix.lower() == ".jsonl":
        return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    return json.loads(path.read_text(encoding="utf-8"))


def find_first(paths, names):
    for name in names:
        hits = list(paths.rglob(name))
        if hits:
            return hits[0]
    return None


def infer_source_paths():
    proposals = repo_path(CONFIG["PROPOSALS_JSON"])
    if proposals is None:
        proposals = find_first(REPO_ROOT / "outputs", ["proposals_normalized.json", "proposals.jsonl"])
    depth_dir = repo_path(CONFIG["DEPTH_DIR"]) or (PREV_DIR / "depth")
    summary_csv = PREV_DIR / "debug" / "depth_refinement_summary.csv"
    return proposals, depth_dir, summary_csv


PROPOSALS_JSON, DEPTH_DIR, PREV_SUMMARY_CSV = infer_source_paths()
if PROPOSALS_JSON is None or not PROPOSALS_JSON.exists():
    raise FileNotFoundError("Set CONFIG['PROPOSALS_JSON']; could not find proposals_normalized.json/proposals.jsonl.")
if not DEPTH_DIR.exists():
    raise FileNotFoundError("Set CONFIG['DEPTH_DIR']; could not find cached depth .npy files.")

proposal_data = read_json_or_jsonl(PROPOSALS_JSON)
if isinstance(proposal_data, dict):
    proposal_data = proposal_data.get("frames") or proposal_data.get("results") or proposal_data.get("data") or []

prev_summary = pd.read_csv(PREV_SUMMARY_CSV) if PREV_SUMMARY_CSV.exists() else pd.DataFrame()

depth_files = list(DEPTH_DIR.rglob("*.npy"))
depth_by_frame = {}
for p in depth_files:
    stem = p.stem
    nums = [int(s) for s in stem.replace("-", "_").split("_") if s.isdigit()]
    if nums:
        depth_by_frame[nums[0]] = p


def norm_bbox(value):
    if value is None:
        return None
    if isinstance(value, str):
        try: value = json.loads(value)
        except Exception: return None
    if isinstance(value, (list, tuple)) and len(value) >= 4:
        return [float(value[0]), float(value[1]), float(value[2]), float(value[3])]
    if isinstance(value, dict) and all(k in value for k in ["x1", "y1", "x2", "y2"]):
        return [float(value[k]) for k in ["x1", "y1", "x2", "y2"]]
    return None


def load_mask(path, shape):
    p = repo_path(path)
    if p is None or not p.exists():
        return None
    m = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if m is None:
        return None
    if m.shape[:2] != shape:
        m = cv2.resize(m, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST)
    return (m > 0).astype(np.uint8)


def bbox_mask(bbox, shape):
    m = np.zeros(shape, dtype=np.uint8)
    if bbox is None:
        return m
    h, w = shape
    x1, y1, x2, y2 = [int(round(v)) for v in bbox]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    if x2 > x1 and y2 > y1:
        m[y1:y2, x1:x2] = 1
    return m


records = []
for i, fr in enumerate(proposal_data):
    frame_idx = int(fr.get("frame_idx", fr.get("extracted_frame_idx", i)))
    depth_path = depth_by_frame.get(frame_idx)
    frame_path = repo_path(fr.get("frame_path") or fr.get("image_path"))
    if frame_path is None or not frame_path.exists() or depth_path is None:
        continue
    img = cv2.imread(str(frame_path))
    if img is None:
        continue
    h, w = img.shape[:2]
    dets = []
    for j, d in enumerate(fr.get("detections") or fr.get("objects") or fr.get("proposals") or []):
        bbox = norm_bbox(d.get("bbox") or d.get("bbox_xyxy") or d.get("box"))
        mask = load_mask(d.get("mask_path"), (h, w))
        if mask is None:
            mask = bbox_mask(bbox, (h, w))
        label = d.get("raw_label") or d.get("label") or d.get("dino_label") or d.get("class_name") or "unknown"
        conf = float(d.get("conf", d.get("score", d.get("confidence", 0.0))) or 0.0)
        det_id = str(d.get("det_id", f"{frame_idx:06d}_{j:04d}"))
        dets.append({"det_id": det_id, "label": str(label), "conf": conf, "bbox": bbox, "mask": mask, "raw_metadata": d})
    if dets:
        records.append({"frame_idx": frame_idx, "frame_path": str(frame_path), "depth_path": str(depth_path), "detections": dets})

if len(records) > CONFIG["SAMPLE_N"]:
    rng = random.Random(CONFIG["RANDOM_SEED"])
    records = sorted(rng.sample(records, CONFIG["SAMPLE_N"]), key=lambda r: r["frame_idx"])

print("frames loaded:", len(records))
print("detections:", sum(len(r["detections"]) for r in records))
print("depth maps:", len({r["depth_path"] for r in records}))
print("previous summary rows:", len(prev_summary))

fig, axes = plt.subplots(min(3, len(records)), 3, figsize=(12, 4 * min(3, len(records))))
axes = np.array(axes).reshape(-1, 3)
for row, rec in zip(axes, records[:3]):
    rgb = cv2.cvtColor(cv2.imread(rec["frame_path"]), cv2.COLOR_BGR2RGB)
    depth = np.load(rec["depth_path"]).astype(np.float32)
    overlay = rgb.copy()
    for d in rec["detections"][:12]:
        overlay[d["mask"].astype(bool)] = (0.6 * overlay[d["mask"].astype(bool)] + 0.4 * np.array([80, 220, 100])).astype(np.uint8)
    row[0].imshow(rgb); row[0].set_title("RGB"); row[0].axis("off")
    row[1].imshow(overlay); row[1].set_title("original mask overlay"); row[1].axis("off")
    row[2].imshow(depth, cmap="viridis"); row[2].set_title("depth"); row[2].axis("off")
plt.tight_layout(); plt.show()

## 3. Utility Functions

In [ ]:
def normalize01(x):
    x = x.astype(np.float32)
    finite = np.isfinite(x)
    if not finite.any():
        return np.zeros_like(x, dtype=np.float32)
    lo, hi = float(np.nanmin(x[finite])), float(np.nanmax(x[finite]))
    if hi <= lo:
        return np.zeros_like(x, dtype=np.float32)
    return np.clip((x - lo) / (hi - lo), 0, 1).astype(np.float32)


def save_mask(mask, path):
    cv2.imwrite(str(path), ((mask > 0).astype(np.uint8) * 255))


def connected_component_count(mask, min_area=1):
    num, labels, stats, _ = cv2.connectedComponentsWithStats((mask > 0).astype(np.uint8), 8)
    return sum(int(stats[i, cv2.CC_STAT_AREA]) >= min_area for i in range(1, num))


def keep_large_components(mask, min_area):
    num, labels, stats, _ = cv2.connectedComponentsWithStats((mask > 0).astype(np.uint8), 8)
    out = np.zeros_like(mask, dtype=np.uint8)
    for i in range(1, num):
        if int(stats[i, cv2.CC_STAT_AREA]) >= min_area:
            out[labels == i] = 1
    return out


def kellipse(k):
    k = max(1, int(k))
    if k % 2 == 0:
        k += 1
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))


def erode_mask(mask, kernel_size):
    return cv2.erode((mask > 0).astype(np.uint8), kellipse(kernel_size), iterations=1)


def dilate_mask(mask, kernel_size):
    return cv2.dilate((mask > 0).astype(np.uint8), kellipse(kernel_size), iterations=1)


def compute_mask_boundary(mask):
    return (cv2.morphologyEx((mask > 0).astype(np.uint8), cv2.MORPH_GRADIENT, kellipse(3)) > 0).astype(np.uint8)


def make_boundary_band(mask, erode_k, dilate_k):
    core = erode_mask(mask, erode_k)
    outer = dilate_mask(mask, dilate_k)
    band = ((outer > 0) & ~(core > 0)).astype(np.uint8)
    return core, outer, band


def safe_area_ratio(candidate_mask, original_mask):
    return float(candidate_mask.sum() / max(1, original_mask.sum()))


def compute_iou(a, b):
    aa, bb = a.astype(bool), b.astype(bool)
    return float((aa & bb).sum() / max(1, (aa | bb).sum()))


def edge_alignment(mask, edge):
    b = compute_mask_boundary(mask)
    near = dilate_mask(b, 5).astype(bool)
    return float(np.mean(edge[near])) if near.any() else 0.0


def candidate_metrics(name, mask, original, edge):
    return {
        f"{name}_area_ratio": safe_area_ratio(mask, original),
        f"{name}_iou": compute_iou(mask, original),
        f"{name}_edge_alignment": edge_alignment(mask, edge),
        f"{name}_components": connected_component_count(mask, CONFIG["MIN_COMPONENT_AREA"]),
    }

## 4. Depth Edge Computation

In [ ]:
def compute_depth_edges(depth, method="sobel"):
    d = normalize01(depth)
    k = CONFIG["DEPTH_EDGE_SMOOTH_KSIZE"]
    if k and k > 1:
        d = cv2.GaussianBlur(d, (k | 1, k | 1), 0)
    if method == "sobel":
        sx = cv2.Sobel(d, cv2.CV_32F, 1, 0, ksize=3)
        sy = cv2.Sobel(d, cv2.CV_32F, 0, 1, ksize=3)
        edge = np.sqrt(sx * sx + sy * sy)
    elif method == "laplacian":
        edge = np.abs(cv2.Laplacian(d, cv2.CV_32F, ksize=3))
    elif method == "canny":
        edge = cv2.Canny((d * 255).astype(np.uint8), 40, 120).astype(np.float32)
    else:
        raise ValueError(method)
    return normalize01(edge)


edge_maps = {}
strong_edges = {}
for rec in records:
    depth = np.load(rec["depth_path"]).astype(np.float32)
    edge = compute_depth_edges(depth, CONFIG["DEPTH_EDGE_METHOD"])
    thr = np.percentile(edge, CONFIG["DEPTH_EDGE_PERCENTILE"])
    edge_maps[rec["frame_idx"]] = edge
    strong_edges[rec["frame_idx"]] = (edge >= thr).astype(np.uint8)
    vis = (plt.cm.magma(edge)[:, :, :3] * 255).astype(np.uint8)
    cv2.imwrite(str(DIRS["debug"] / f"depth_edge_{rec['frame_idx']:06d}.png"), cv2.cvtColor(vis, cv2.COLOR_RGB2BGR))

show = records[:min(3, len(records))]
fig, axes = plt.subplots(len(show), 4, figsize=(15, 4 * len(show)))
axes = np.array(axes).reshape(len(show), 4)
for row, rec in zip(axes, show):
    rgb = cv2.cvtColor(cv2.imread(rec["frame_path"]), cv2.COLOR_BGR2RGB)
    depth = np.load(rec["depth_path"])
    row[0].imshow(rgb); row[0].set_title("RGB"); row[0].axis("off")
    row[1].imshow(depth, cmap="viridis"); row[1].set_title("depth"); row[1].axis("off")
    row[2].imshow(edge_maps[rec["frame_idx"]], cmap="magma"); row[2].set_title("depth edge"); row[2].axis("off")
    row[3].imshow(strong_edges[rec["frame_idx"]], cmap="gray"); row[3].set_title("strong edge"); row[3].axis("off")
plt.tight_layout(); plt.show()

## 5. Boundary Band and Edge Alignment Metrics

In [ ]:
def boundary_metrics(mask, depth, edge):
    core, outer, band = make_boundary_band(mask, CONFIG["CORE_ERODE_KERNEL"], CONFIG["BAND_DILATE_KERNEL"])
    boundary = compute_mask_boundary(mask)
    inside_shell = ((mask > 0) & ~(core > 0)).astype(np.uint8)
    outside_shell = ((outer > 0) & ~(mask > 0)).astype(np.uint8)
    vals = depth[mask.astype(bool)]
    core_vals = depth[core.astype(bool)]
    ref = core_vals if core_vals.size >= CONFIG["MIN_COMPONENT_AREA"] else vals
    med = float(np.median(ref)) if ref.size else 0.0
    outlier = ((inside_shell > 0) & (np.abs(depth - med) > CONFIG["BOUNDARY_DEPTH_OUTLIER_THRESHOLD"])).astype(np.uint8)
    return {
        "mask_area": int(mask.sum()),
        "core_area_ratio": float(core.sum() / max(1, mask.sum())),
        "boundary_band_area": int(band.sum()),
        "original_component_count": connected_component_count(mask, CONFIG["MIN_COMPONENT_AREA"]),
        "depth_std_inside_mask": float(np.std(vals)) if vals.size else np.nan,
        "depth_range_p80_p20_inside_mask": float(np.percentile(vals, 80) - np.percentile(vals, 20)) if vals.size else np.nan,
        "boundary_edge_alignment": edge_alignment(mask, edge),
        "band_edge_strength": float(np.mean(edge[band.astype(bool)])) if band.any() else 0.0,
        "outside_near_edge_strength": float(np.mean(edge[outside_shell.astype(bool)])) if outside_shell.any() else 0.0,
        "inside_near_edge_strength": float(np.mean(edge[inside_shell.astype(bool)])) if inside_shell.any() else 0.0,
        "boundary_depth_outlier_ratio": float(outlier.sum() / max(1, inside_shell.sum())),
    }


metric_rows = []
for rec in records:
    depth = np.load(rec["depth_path"]).astype(np.float32)
    edge = edge_maps[rec["frame_idx"]]
    for d in rec["detections"]:
        m = boundary_metrics(d["mask"], depth, edge)
        d["boundary_metrics"] = m
        metric_rows.append({"frame_idx": rec["frame_idx"], "det_id": d["det_id"], "label": d["label"], "conf": d["conf"], **m})
boundary_metrics_df = pd.DataFrame(metric_rows)
boundary_metrics_df.to_csv(DIRS["debug"] / "boundary_edge_metrics.csv", index=False)
display(boundary_metrics_df.sort_values(["boundary_edge_alignment", "boundary_depth_outlier_ratio"], ascending=[True, False]).head(30))

## 6-12. Candidate Methods and Conservative Final Decision

In [ ]:
def aggressive_baseline(mask, depth):
    vals = depth[mask.astype(bool)]
    if vals.size == 0:
        return mask.copy(), "aggressive_empty"
    med = float(np.median(vals))
    cand = ((mask > 0) & (np.abs(depth - med) < CONFIG["BOUNDARY_DEPTH_OUTLIER_THRESHOLD"])).astype(np.uint8)
    cand = keep_large_components(cand, CONFIG["MIN_COMPONENT_AREA"])
    return cand, "aggressive_baseline_not_for_final_use"


def safety(candidate, original, core, edge, before_align):
    area_ratio = safe_area_ratio(candidate, original)
    comp_before = connected_component_count(original, CONFIG["MIN_COMPONENT_AREA"])
    comp_after = connected_component_count(candidate, CONFIG["MIN_COMPONENT_AREA"])
    core_preserved = ((candidate > 0) | ~(core > 0)).all()
    align = edge_alignment(candidate, edge)
    return {
        "area_ratio": area_ratio,
        "iou": compute_iou(candidate, original),
        "component_count_before": comp_before,
        "component_count_after": comp_after,
        "component_count_change": comp_after - comp_before,
        "core_preserved": bool(core_preserved),
        "edge_alignment": align,
        "alignment_gain": align - before_align,
        "safety_passed": bool(core_preserved and area_ratio >= 1 - CONFIG["MAX_ALLOWED_AREA_DROP"] and comp_after <= comp_before + CONFIG["MAX_ALLOWED_COMPONENT_INCREASE"]),
    }


def boundary_only_depth_trim(mask, depth, edge):
    core = erode_mask(mask, CONFIG["CORE_ERODE_KERNEL"])
    if core.sum() < CONFIG["MIN_COMPONENT_AREA"]:
        core = mask.copy()
    shell = ((mask > 0) & ~(core > 0)).astype(np.uint8)
    vals = depth[core.astype(bool)] if core.sum() else depth[mask.astype(bool)]
    med = float(np.median(vals)) if vals.size else 0.0
    outlier = ((shell > 0) & (np.abs(depth - med) > CONFIG["BOUNDARY_DEPTH_OUTLIER_THRESHOLD"])).astype(np.uint8)
    cand = ((core > 0) | ((shell > 0) & ~(outlier > 0))).astype(np.uint8)
    cand = cv2.morphologyEx(cand, cv2.MORPH_CLOSE, kellipse(3), iterations=1)
    cand = ((cand > 0) | (core > 0)).astype(np.uint8)
    cand = keep_large_components(cand, CONFIG["MIN_COMPONENT_AREA"])
    cand = ((cand > 0) | (core > 0)).astype(np.uint8)
    removed_ratio = float(outlier.sum() / max(1, mask.sum()))
    before_align = edge_alignment(mask, edge)
    s = safety(cand, mask, core, edge, before_align)
    passed = s["safety_passed"] and removed_ratio <= CONFIG["BOUNDARY_TRIM_MAX_REMOVED_RATIO"] and s["area_ratio"] >= CONFIG["BOUNDARY_TRIM_MIN_FINAL_AREA_RATIO"]
    return (cand if passed else mask.copy()), {"action": "boundary_trim_accepted" if passed else "trim_rejected_keep_original", "removed_ratio": removed_ratio, **s}


def edge_snap_candidate(mask, depth, edge, strong):
    core, outer, band = make_boundary_band(mask, CONFIG["CORE_ERODE_KERNEL"], CONFIG["BAND_DILATE_KERNEL"])
    if core.sum() < CONFIG["MIN_COMPONENT_AREA"]:
        core = mask.copy()
    shell = ((mask > 0) & ~(core > 0)).astype(np.uint8)
    # Conservative approximation: keep shell pixels supported by nearby strong depth edges or close to core depth.
    near_edge = cv2.dilate(strong.astype(np.uint8), kellipse(CONFIG["SNAP_SEARCH_RADIUS"] * 2 + 1), iterations=1)
    vals = depth[core.astype(bool)] if core.sum() else depth[mask.astype(bool)]
    med = float(np.median(vals)) if vals.size else 0.0
    depth_ok = np.abs(depth - med) <= CONFIG["BOUNDARY_DEPTH_OUTLIER_THRESHOLD"] * 1.25
    selected_shell = ((shell > 0) & ((near_edge > 0) | depth_ok)).astype(np.uint8)
    cand = ((core > 0) | (selected_shell > 0)).astype(np.uint8)
    cand = keep_large_components(cand, CONFIG["MIN_COMPONENT_AREA"])
    cand = ((cand > 0) | (core > 0)).astype(np.uint8)
    before_align = edge_alignment(mask, edge)
    s = safety(cand, mask, core, edge, before_align)
    area_change = abs(1 - s["area_ratio"])
    passed = s["safety_passed"] and area_change <= CONFIG["SNAP_MAX_AREA_CHANGE_RATIO"] and s["alignment_gain"] >= CONFIG["SNAP_MIN_EDGE_ALIGNMENT_GAIN"]
    return (cand if passed else mask.copy()), {"action": "edge_snap_candidate" if passed else "edge_snap_rejected_keep_original", "area_change_ratio": area_change, **s}


def rgbd_grabcut_candidate(image_rgb, depth, mask, edge):
    core, outer, band = make_boundary_band(mask, CONFIG["CORE_ERODE_KERNEL"], CONFIG["BAND_DILATE_KERNEL"])
    if core.sum() < CONFIG["MIN_COMPONENT_AREA"]:
        core = erode_mask(mask, 3)
    gc = np.full(mask.shape, cv2.GC_BGD, dtype=np.uint8)
    gc[outer.astype(bool)] = cv2.GC_PR_BGD
    gc[mask.astype(bool)] = cv2.GC_PR_FGD
    gc[core.astype(bool)] = cv2.GC_FGD
    bgd, fgd = np.zeros((1,65), np.float64), np.zeros((1,65), np.float64)
    try:
        cv2.grabCut(cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR), gc, None, bgd, fgd, CONFIG["GRABCUT_ITER"], cv2.GC_INIT_WITH_MASK)
        cand = np.where((gc == cv2.GC_FGD) | (gc == cv2.GC_PR_FGD), 1, 0).astype(np.uint8)
        cand = ((cand > 0) & (outer > 0)).astype(np.uint8)
        cand = ((cand > 0) | (core > 0)).astype(np.uint8)
        cand = keep_large_components(cand, CONFIG["MIN_COMPONENT_AREA"])
        cand = ((cand > 0) | (core > 0)).astype(np.uint8)
        before_align = edge_alignment(mask, edge)
        s = safety(cand, mask, core, edge, before_align)
        passed = s["safety_passed"] and s["iou"] >= 0.60
        return (cand if passed else mask.copy()), {"action": "grabcut_candidate" if passed else "grabcut_rejected_keep_original", **s}
    except Exception as exc:
        return mask.copy(), {"action": "grabcut_error_keep_original", "error": str(exc), **safety(mask, mask, core, edge, edge_alignment(mask, edge))}


def select_final(original, candidates, metrics):
    if metrics["boundary_edge_alignment"] >= CONFIG["MIN_EDGE_ALIGNMENT_FOR_ACCEPT"]:
        return original, "accept_original", False, "original boundary already aligns with depth edge"
    for method in ["boundary_trim", "edge_snap", "grabcut"]:
        cand, meta = candidates[method]
        if meta.get("safety_passed") and meta.get("action", "").endswith("accepted") or meta.get("action") in {"edge_snap_candidate", "grabcut_candidate"}:
            return cand, f"use_{method}", True, meta.get("action", "candidate passed safety")
    return original, "keep_original_review", False, "no candidate passed conservative safety checks"


rows = []
for rec in tqdm(records, desc="boundary refinement candidates"):
    image_rgb = cv2.cvtColor(cv2.imread(rec["frame_path"]), cv2.COLOR_BGR2RGB)
    depth = np.load(rec["depth_path"]).astype(np.float32)
    edge = edge_maps[rec["frame_idx"]]
    strong = strong_edges[rec["frame_idx"]]
    for d in rec["detections"]:
        mask = d["mask"]
        if mask.sum() < CONFIG["MIN_MASK_AREA"]:
            continue
        aggr, aggr_action = aggressive_baseline(mask, depth)
        btrim, bmeta = boundary_only_depth_trim(mask, depth, edge)
        snap, smeta = edge_snap_candidate(mask, depth, edge, strong)
        grab, gmeta = rgbd_grabcut_candidate(image_rgb, depth, mask, edge) if CONFIG["ENABLE_GRABCUT_CANDIDATE"] else (mask.copy(), {"action": "grabcut_disabled"})
        final, selected, replace, reason = select_final(mask, {"boundary_trim": (btrim,bmeta), "edge_snap": (snap,smeta), "grabcut": (grab,gmeta)}, d["boundary_metrics"])
        d.update({"aggressive_mask": aggr, "boundary_trim_mask": btrim, "edge_snap_mask": snap, "grabcut_mask": grab, "final_mask": final, "selected_method": selected, "replace_mask": replace, "reason": reason, "candidate_meta": {"boundary_trim": bmeta, "edge_snap": smeta, "grabcut": gmeta}})
        for name, m, folder in [("original", mask, "masks_original"), ("aggressive", aggr, "masks_aggressive_baseline"), ("boundary_trim", btrim, "masks_boundary_trim"), ("edge_snap", snap, "masks_edge_snap"), ("grabcut", grab, "masks_grabcut"), ("final", final, "masks_final")]:
            save_mask(m, DIRS[folder] / f"frame_{rec['frame_idx']:06d}_{d['det_id'].replace('/','_')}_{name}.png")
        comp_before = connected_component_count(mask, CONFIG["MIN_COMPONENT_AREA"])
        comp_after = connected_component_count(final, CONFIG["MIN_COMPONENT_AREA"])
        row = {
            "frame_idx": rec["frame_idx"], "det_id": d["det_id"], "label": d["label"], "conf": d["conf"],
            "original_area": int(mask.sum()),
            **candidate_metrics("aggressive", aggr, mask, edge),
            **candidate_metrics("boundary_trim", btrim, mask, edge),
            **candidate_metrics("edge_snap", snap, mask, edge),
            **candidate_metrics("grabcut", grab, mask, edge),
            "original_edge_alignment": d["boundary_metrics"]["boundary_edge_alignment"],
            "selected_method": selected, "replace_mask": replace,
            "final_area_ratio": safe_area_ratio(final, mask),
            "final_iou_with_original": compute_iou(final, mask),
            "component_count_before": comp_before,
            "component_count_after": comp_after,
            "safety_passed": bool(replace or selected == "accept_original"),
            "review_flag": selected == "keep_original_review",
            "reason": reason,
        }
        rows.append(row)

summary_df = pd.DataFrame(rows)
summary_df.to_csv(DIRS["debug"] / "boundary_refinement_summary.csv", index=False)
display(summary_df.head())

## 13. Visualizations

In [ ]:
def overlay(image, mask, color, alpha=0.45):
    out = image.copy()
    sel = mask.astype(bool)
    out[sel] = (1 - alpha) * out[sel] + alpha * np.array(color)
    return out.astype(np.uint8)


def draw_masks(rec, key):
    image = cv2.cvtColor(cv2.imread(rec["frame_path"]), cv2.COLOR_BGR2RGB)
    out = image.copy()
    for i, d in enumerate(rec["detections"][:20]):
        if key not in d:
            continue
        color = [(70+i*47)%255, (180+i*31)%255, (90+i*67)%255]
        out = overlay(out, d[key], color, 0.35)
        if d.get("bbox"):
            x1,y1,x2,y2 = [int(v) for v in d["bbox"]]
            cv2.rectangle(out, (x1,y1), (x2,y2), color, 2)
            cv2.putText(out, f"{d['label']} {d.get('selected_method','')}"[:50], (x1, max(15,y1-4)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)
    return out


def edge_boundary_view(rec):
    image = cv2.cvtColor(cv2.imread(rec["frame_path"]), cv2.COLOR_BGR2RGB)
    edge_rgb = (plt.cm.magma(edge_maps[rec["frame_idx"]])[:,:,:3] * 255).astype(np.uint8)
    out = (0.4 * image + 0.6 * edge_rgb).astype(np.uint8)
    for d in rec["detections"][:20]:
        b = compute_mask_boundary(d["mask"]).astype(bool)
        out[b] = [50, 255, 80]
    return out


def diff_view(rec):
    image = cv2.cvtColor(cv2.imread(rec["frame_path"]), cv2.COLOR_BGR2RGB)
    out = image.copy()
    diff = np.zeros_like(out)
    for d in rec["detections"][:20]:
        orig = d["mask"].astype(bool)
        final = d.get("final_mask", d["mask"]).astype(bool)
        diff[orig & final] = [40, 210, 80]
        diff[orig & ~final] = [240, 50, 50]
        diff[final & ~orig] = [50, 120, 240]
    return (0.55 * out + 0.45 * diff).astype(np.uint8)


vis_paths = []
for rec in records:
    depth = np.load(rec["depth_path"])
    fig, axes = plt.subplots(3, 3, figsize=(21, 15))
    axes = axes.ravel()
    axes[0].imshow(draw_masks(rec, "mask")); axes[0].set_title("A Original SAM2"); axes[0].axis("off")
    axes[1].imshow(depth, cmap="viridis"); axes[1].set_title("B Depth"); axes[1].axis("off")
    axes[2].imshow(edge_boundary_view(rec)); axes[2].set_title("C Depth edge + mask boundary"); axes[2].axis("off")
    axes[3].imshow(draw_masks(rec, "aggressive_mask")); axes[3].set_title("D Aggressive baseline only"); axes[3].axis("off")
    axes[4].imshow(draw_masks(rec, "boundary_trim_mask")); axes[4].set_title("E Boundary trim"); axes[4].axis("off")
    axes[5].imshow(draw_masks(rec, "edge_snap_mask")); axes[5].set_title("F Edge snap"); axes[5].axis("off")
    axes[6].imshow(draw_masks(rec, "grabcut_mask")); axes[6].set_title("G GrabCut"); axes[6].axis("off")
    axes[7].imshow(draw_masks(rec, "final_mask")); axes[7].set_title("H Conservative final"); axes[7].axis("off")
    axes[8].imshow(diff_view(rec)); axes[8].set_title("I kept green / removed red / added blue"); axes[8].axis("off")
    fig.suptitle(f"frame {rec['frame_idx']}")
    plt.tight_layout()
    out = DIRS["visualizations"] / f"frame_{rec['frame_idx']:06d}_boundary_refinement.png"
    if CONFIG["SAVE_VIS"]:
        fig.savefig(out, dpi=140)
    if CONFIG["SHOW_INLINE"]:
        plt.show()
    else:
        plt.close(fig)
    vis_paths.append(out)


def make_contact_sheet(paths, out_path, thumb_w=520):
    imgs = []
    for p in paths:
        im = cv2.imread(str(p))
        if im is None: continue
        h,w = im.shape[:2]
        imgs.append(cv2.resize(im, (thumb_w, int(h * thumb_w / w)), interpolation=cv2.INTER_AREA))
    if not imgs: return None
    cols = 2; rows = math.ceil(len(imgs)/cols); th = max(i.shape[0] for i in imgs)
    canvas = np.full((rows*th, cols*thumb_w, 3), 245, dtype=np.uint8)
    for i, im in enumerate(imgs):
        r,c = divmod(i, cols); canvas[r*th:r*th+im.shape[0], c*thumb_w:c*thumb_w+im.shape[1]] = im
    cv2.imwrite(str(out_path), canvas); return out_path

if CONFIG["SAVE_CONTACT_SHEET"]:
    print("contact sheet:", make_contact_sheet(vis_paths, DIRS["visualizations"] / "boundary_refinement_contact_sheet.jpg"))

## 14. Quantitative Summary

In [ ]:
print("summary path:", DIRS["debug"] / "boundary_refinement_summary.csv")
print("total detections:", len(summary_df))
print(summary_df["selected_method"].value_counts())
print("replace masks:", int(summary_df["replace_mask"].sum()))
print()
print("Average area change by selected method:")
display(summary_df.assign(area_change=lambda x: abs(1 - x["final_area_ratio"])).groupby("selected_method")["area_change"].mean().sort_values())
print("Average edge alignment by selected method:")
display(summary_df.groupby("selected_method")[["original_edge_alignment", "boundary_trim_edge_alignment", "edge_snap_edge_alignment", "grabcut_edge_alignment"]].mean())
print("Largest aggressive area drop:")
display(summary_df.sort_values("aggressive_area_ratio").head(20))
print("Best safe edge alignment gain:")
gain = summary_df.assign(final_gain=lambda x: np.where(x["selected_method"]=="use_boundary_trim", x["boundary_trim_edge_alignment"]-x["original_edge_alignment"], np.where(x["selected_method"]=="use_edge_snap", x["edge_snap_edge_alignment"]-x["original_edge_alignment"], x["grabcut_edge_alignment"]-x["original_edge_alignment"])))
display(gain.sort_values("final_gain", ascending=False).head(20))
print("Review needed:")
display(summary_df[summary_df["review_flag"]].head(30))

## 15. Feasibility Assessment

In [ ]:
total = max(1, len(summary_df))
replace_ratio = summary_df["replace_mask"].mean()
review_ratio = summary_df["review_flag"].mean()
aggressive_damaged = (summary_df["aggressive_area_ratio"] < 0.60).sum()
if replace_ratio > 0 and replace_ratio < 0.30 and aggressive_damaged > 0:
    decision = "Feasibility: promising. Depth edges are useful as conservative boundary cues."
elif review_ratio > 0.25 or replace_ratio > 0:
    decision = "Feasibility: mixed. Depth edge is informative, but replacement policy must stay conservative."
else:
    decision = "Feasibility: use depth edges only for review/scoring, not replacement."
print(decision)
print()
print("Recommendations:")
print("- If aggressive baseline damages masks, do not use median-depth hard thresholding.")
print("- If edge alignment is high for original masks, keep original masks.")
print("- If boundary trim helps only slightly, use it as optional cleanup, not default.")
print("- If GrabCut works well, consider RGB-D GrabCut as a candidate branch.")
print("- If close objects still merge, use split-candidate review rather than automatic split.")
print("- If depth edge is clean but masks are not improved, use depth edge as quality/review signal for training crop export.")

## 16. Important Constraints

- Keep original SAM2 masks as the default.
- Do not aggressively delete mask interiors.
- Do not use depth median thresholding as final mask logic.
- Use depth edge only around mask boundary.
- Prefer review flags over unsafe automatic replacement.
- The functions here can later be moved into `depth_edge_boundary_refinement.py`.